In [ ]:

# 安装 condacolab 并重启内核
!pip install -q condacolab
import condacolab
condacolab.install()


In [ ]:
# 安装 augustus（包括物种模型）
!conda install -c bioconda augustus -y

In [ ]:
import os
import subprocess

# 查找 augustus config 目录
conda_prefix = os.environ.get('CONDA_PREFIX', '/usr/local')
result = subprocess.run(['find', conda_prefix, '-type', 'd', '-name', 'config', '-path', '*/augustus/*'],
                        capture_output=True, text=True)
config_path = result.stdout.strip().split('\n')[0] if result.stdout else None

if config_path and os.path.exists(config_path):
    os.environ['AUGUSTUS_CONFIG_PATH'] = config_path
    print(f"✅ AUGUSTUS_CONFIG_PATH 设置为: {config_path}")
else:
    # 尝试常见备选路径
    alt_paths = ['/opt/augustus/config', '/usr/local/share/augustus/config', '/usr/share/augustus/config']
    for p in alt_paths:
        if os.path.exists(p):
            os.environ['AUGUSTUS_CONFIG_PATH'] = p
            print(f"✅ 使用备选路径: {p}")
            break
    else:
        raise Exception("无法找到 augustus config 目录，请手动设置")

# 验证 augustus 可用
!augustus --version
!augustus --species=help | grep -i human

In [ ]:
# 下载 Ensembl release-112 的 chr21 序列
!wget -O chr21.fa.gz https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.21.fa.gz
!gunzip -f chr21.fa.gz
!head -n 2 chr21.fa

In [ ]:
# 运行预测（耗时约 10-20 分钟）
!augustus --species=human --gff3=on --UTR=on chr21.fa > chr21_augustus.gff3
print("预测完成！")

In [ ]:
# 统计 gene 和 transcript 行数
import subprocess

def count_features(gff_file, feature_type):
    cmd = f"grep -c $'\t{feature_type}\t' {gff_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return int(result.stdout.strip())

genes_chr21 = count_features('chr21_augustus.gff3', 'gene')
transcripts_chr21 = count_features('chr21_augustus.gff3', 'transcript')
# 有些版本用 'mRNA' 表示转录本
if transcripts_chr21 == 0:
    transcripts_chr21 = count_features('chr21_augustus.gff3', 'mRNA')

print(f"chr21 预测结果：基因数 = {genes_chr21}, 转录本数 = {transcripts_chr21}")

In [ ]:
# GRCh38 primary assembly 总大小（bp）
# 数据来源：https://www.ncbi.nlm.nih.gov/assembly/GCF_000001405.40/
total_genome_size = 3_099_706_404  # 约 3.1 Gb

# chr21 的大小（从下载的 fasta 统计）
!samtools faidx chr21.fa 2>/dev/null || (conda install -c bioconda samtools -y && samtools faidx chr21.fa)
chr21_size = int(subprocess.run("cut -f2 chr21.fa.fai", shell=True, capture_output=True, text=True).stdout.strip())

ratio = total_genome_size / chr21_size
estimated_genes_total = int(genes_chr21 * ratio)
estimated_transcripts_total = int(transcripts_chr21 * ratio)

print(f"chr21 大小: {chr21_size:,} bp")
print(f"全基因组大小: {total_genome_size:,} bp")
print(f"比例因子: {ratio:.2f}")
print(f"\n📊 基于 AUGUSTUS 预测的估计：")
print(f"全基因组基因数 ≈ {estimated_genes_total:,}")
print(f"全基因组转录本数 ≈ {estimated_transcripts_total:,}")